# Evaluasi dan Analisis 


In [ ]:
import os
import sys
import json
import time
import random
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import defaultdict
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
from nltk.translate.meteor_score import meteor_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.abspath(""), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.rnn.RNNKeras import RNNKeras
from src.rnn.RNNScratch import RNNScratch
from src.lstm.lstm_decoder import LSTMDecoder # Sesuaikan lagi LSTM nya

metadata_path = os.path.join(PROJECT_ROOT, "outputs", "vocab", "metadata.json")
vocab_path = os.path.join(PROJECT_ROOT, "outputs", "vocab", "vocab.json")
test_txt = os.path.join(PROJECT_ROOT, "data", "Flickr_8k.testImages.txt")
images_dir = os.path.join(PROJECT_ROOT, "Flickr8k", "Images")
captions_file = os.path.join(PROJECT_ROOT, "Flickr8k", "captions.txt")
models_dir = os.path.join(PROJECT_ROOT, "models")
output_dir = os.path.join(PROJECT_ROOT, "outputs")

with open(vocab_path, "r") as f:
    word_to_idx = json.load(f)
idx_to_word = {str(idx): word for word, idx in word_to_idx.items()}

with open(test_txt, "r") as f:
    test_images = [line.strip() for line in f if line.strip()]
sample_images = [img for img in test_images if os.path.exists(os.path.join(images_dir, img))]

def load_captions(captions_file):
    image_captions = {}
    with open(captions_file, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(",", 1)
            if len(parts) == 2:
                img, cap = parts
                if img not in image_captions:
                    image_captions[img] = []
                image_captions[img].append(cap)
    return image_captions

image_captions = load_captions(captions_file)

eval_images = sample_images[:100] # 100 untuk quick test, ganti ke seluruh test set untuk final
references = []
meteor_references = []
for img in eval_images:
    img_refs = []
    meteor_refs = []
    for cap in image_captions[img]:
        words = cap.lower().replace(".", "").replace(",", "").split()
        img_refs.append(words)
        meteor_refs.append(" ".join(words))
    references.append(img_refs)
    meteor_references.append(meteor_refs)


## 1. Variasi Jumlah Layer dan Hidden State
Membandingkan grafik *training loss* dan *validation loss*, serta menghitung BLEU-4 dan METEOR untuk setiap variasi.

In [ ]:
def plot_history(architecture):
    arch_dir = os.path.join(models_dir, architecture)
    if not os.path.exists(arch_dir):
        print(f"Folder {arch_dir} belum ada. Model mungkin belum dilatih.")
        return
        
    histories = glob.glob(os.path.join(arch_dir, "*", "history.json"))
    if not histories:
        print(f"Belum ada file history.json di dalam {arch_dir}.")
        return
        
    plt.figure(figsize=(15, 10))
    
    for i, hist_file in enumerate(sorted(histories)):
        config_name = os.path.basename(os.path.dirname(hist_file))
        with open(hist_file, 'r') as f:
            h = json.load(f)
            
        plt.subplot(2, 3, i+1)
        plt.plot(h['loss'], label='Train Loss')
        plt.plot(h['val_loss'], label='Val Loss')
        plt.title(f"{architecture.upper()} - {config_name}")
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
    plt.tight_layout()
    plt.show()

print("--- Grafik Loss RNN ---")
plot_history("rnn")

print("\n--- Grafik Loss LSTM ---")
plot_history("lstm")


In [ ]:
# Evaluasi BLEU dan METEOR
def evaluate_all_variations(architecture):
    arch_dir = os.path.join(models_dir, architecture)
    if not os.path.exists(arch_dir): return []
    
    configs = [d for d in os.listdir(arch_dir) if os.path.isdir(os.path.join(arch_dir, d))]
    results = []
    
    for config in sorted(configs):
        model_path = os.path.join(arch_dir, config, f"{config}.keras")
        if not os.path.exists(model_path): continue
            
        print(f"Mengevaluasi {architecture.upper()} - {config}...")
        
        try:
            captioner = RNNScratch(model_path, metadata_path) 
            # Sesuaikan lagi untuk LSTM
        except:
            print(f"Gagal memuat {config}.")
            continue
            
        hypotheses = []
        meteor_scores = []
        start_time = time.time()
        
        for i, img in enumerate(eval_images):
            img_path = os.path.join(images_dir, img)
            caption = captioner.generate_caption(img_path, idx_to_word)
            
            words = [w for w in caption.split() if w not in ["<start>", "<end>", "<pad>"]]
            hypotheses.append(words)
            
            # Hitung METEOR per kalimat lalu dirata-rata
            m_score = meteor_score(meteor_references[i], words)
            meteor_scores.append(m_score)
            
        total_time = time.time() - start_time
        bleu4 = corpus_bleu(references, hypotheses)
        avg_meteor = np.mean(meteor_scores)
        
        results.append({
            "config": config,
            "bleu4": bleu4,
            "meteor": avg_meteor,
            "time_s": total_time / len(eval_images)
        })
    return results

# saat ini evaluasi menggunakan sample gambar test (masih 100). Jika ingin sesuaikan lagi list eval_images 
rnn_results = evaluate_all_variations("rnn")
lstm_results = evaluate_all_variations("lstm") # Sesuaikan lagi LSTM nya

def print_res(res, name):
    print(f"\n--- Hasil {name.upper()} ---")
    print(f"{'Config':<15} | {'BLEU-4':<8} | {'METEOR':<8} | {'Time/Img':<8}")
    for r in res:
        print(f"{r['config']:<15} | {r['bleu4']:<8.4f} | {r['meteor']:<8.4f} | {r['time_s']:<8.4f}s")

if rnn_results: print_res(rnn_results, "rnn")
if lstm_results: print_res(lstm_results, "lstm")


**Kesimpulan Pengaruh Jumlah Layer dan Hidden State:**
- XXX
- YYY

## 2. Perbandingan Keras vs Scratch
Membandingkan skor dan waktu eksekusi antara implementasi *Native Keras* dan *From Scratch* untuk model terbaik.

In [ ]:
# Pilih model terbaik
best_model_name = "rnn_L1_H128" # Ganti sesuai model terbaik
best_model_path = os.path.join(models_dir, "rnn", best_model_name, f"{best_model_name}.keras")

if os.path.exists(best_model_path):
    keras_model = RNNKeras(best_model_path, metadata_path)
    scratch_model = RNNScratch(best_model_path, metadata_path)
    
    for name, model in [("Keras", keras_model), ("Scratch", scratch_model)]:
        hypotheses = []
        start_time = time.time()
        for img in eval_images:
            img_path = os.path.join(images_dir, img)
            caption = model.generate_caption(img_path, idx_to_word)
            words = [w for w in caption.split() if w not in ["<start>", "<end>", "<pad>"]]
            hypotheses.append(words)
            
        t = (time.time() - start_time) / len(eval_images)
        b4 = corpus_bleu(references, hypotheses)
        print(f"[{name}] BLEU-4: {b4:.4f} | Waktu rata-rata per gambar: {t:.4f} detik")
else:
    print("Set model path yang benar di variabel best_model_name!")


**Kesimpulan Perbandingan Keras vs Scratch:**
- XXX
- YYY

## 3. Perbandingan RNN vs LSTM

In [ ]:
if 'rnn_results' in locals() and 'lstm_results' in locals() and rnn_results and lstm_results:
    #  Cari model terbaik berdasarkan skor BLEU-4 tertinggi
    best_rnn = max(rnn_results, key=lambda x: x["bleu4"])
    best_lstm = max(lstm_results, key=lambda x: x["bleu4"])
    
    print("="*60)
    print("PERBANDINGAN ARSITEKTUR TERBAIK: RNN vs LSTM")
    print("="*60)
    
    print(f"RNN TERBAIK: {best_rnn['config']}")
    print(f" - BLEU-4 Score          : {best_rnn['bleu4']:.4f}")
    print(f" - METEOR Score          : {best_rnn['meteor']:.4f}")
    print(f" - Waktu Inferensi / Img : {best_rnn['time_s']:.4f} detik")
    print()
    
    print(f"LSTM TERBAIK: {best_lstm['config']}")
    print(f" - BLEU-4 Score          : {best_lstm['bleu4']:.4f}")
    print(f" - METEOR Score          : {best_lstm['meteor']:.4f}")
    print(f" - Waktu Inferensi / Img : {best_lstm['time_s']:.4f} detik")
    print("="*60)
    
    # selisih
    diff_bleu = best_lstm['bleu4'] - best_rnn['bleu4']
    diff_time = best_lstm['time_s'] - best_rnn['time_s']
    
    if diff_bleu > 0:
        print(f"LSTM mengungguli RNN sebesar +{diff_bleu:.4f} poin BLEU-4.")
    else:
        print(f"RNN mengungguli LSTM sebesar +{abs(diff_bleu):.4f} poin BLEU-4.")
        
    if diff_time > 0:
        print(f"LSTM lebih lambat {diff_time:.4f} detik per gambar dibandingkan RNN.")
    else:
        print(f"LSTM lebih cepat {abs(diff_time):.4f} detik per gambar dibandingkan RNN.")
else:
    print("Data evaluasi belum tersedia. Harap latih dan evaluasi kedua model terlebih dahulu.")


In [ ]:
if rnn_results and lstm_results:
    best_rnn_path = os.path.join(models_dir, "rnn", "rnn_L1_H128", "rnn_L1_H128.keras") # Sesuaikan model terbaik
    best_lstm_path = os.path.join(models_dir, "lstm", "lstm_L1_H128", "lstm_L1_H128.keras") # Sesuaikan model terbaik
    
    if os.path.exists(best_rnn_path) and os.path.exists(best_lstm_path):
        rnn_model = RNNScratch(best_rnn_path, metadata_path)
        # lstm_model = LSTMDecoder() # Ganti dengan class scratch LSTM yang benar
        
        # Simulasi Kualitatif (Tampilkan 10 Gambar dengan variasi skor)
        # ... (Tulis loop untuk generate gambar, hitung sentence_bleu, dan tampilkan subplot)
        print("Model berhasil diload! Silakan tambahkan kode subplot untuk 10 gambar di sini.")
else:
    print("Harap latih model LSTM terlebih dahulu untuk perbandingan ini.")


**Analisis Kualitatif & Teori Vanishing Gradient:**
- XXX
- YYY

## 4. Pengaruh Panjang Maksimum Caption

In [ ]:
if os.path.exists(best_model_path):
    keras_model = RNNKeras(best_model_path, metadata_path) # menggunakan Keras agar lebih cepat, sesuaikan lagi jika perlu menggunakan from scratch
    
    max_lengths = [10, 20, 30]
    for max_len in max_lengths:
        hypotheses = []
        for img in eval_images:
            img_path = os.path.join(images_dir, img)
            caption = keras_model.generate_caption(img_path, idx_to_word, max_len=max_len)
            words = [w for w in caption.split() if w not in ["<start>", "<end>", "<pad>"]]
            hypotheses.append(words)
            
        b4 = corpus_bleu(references, hypotheses)
        print(f"Max Length {max_len} | BLEU-4: {b4:.4f}")


**Kesimpulan Pengaruh Panjang Maksimum:**
- XXX
- YYY